# NLQuery - Natural Language Table Question Answering
## Section 1: Setup, Data Pipeline & Drive Caching

**Team:** Vrushabh, Jeni Savalia, Mahak Gupta

**Project Overview:**
- Dataset: WikiTableQuestions (22K QA pairs, 2,108 Wikipedia tables)
- Task: Natural Language -> SQL -> Execute -> Answer
- Approaches: Prompt Engineering, RAG, Fine-Tuning

---

### How This Notebook Works
- **First run:** Cells download data, process tables, build SQLite DBs, and cache everything to Google Drive.
- **Subsequent runs:** Cells detect cached files on Drive and skip expensive processing.
- **Rule:** Always run cells in order from top to bottom on a fresh session.

---
## 1.1 Install Dependencies

In [1]:
%%capture
!pip install datasets==2.14.7 transformers==4.46.0 huggingface_hub==0.26.0 accelerate bitsandbytes sentence-transformers faiss-gpu peft trl gradio sqlparse
print("All dependencies installed!")

---
## 1.2 Imports

In [2]:
import os
import json
import sqlite3
import pickle
import time
import re
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from collections import Counter, defaultdict
from tqdm.notebook import tqdm

from datasets import load_dataset

print("All imports loaded successfully!")

All imports loaded successfully!


---
## 1.3 Mount Google Drive & Create Folder Structure

In [3]:
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/NLQuery'

subfolders = [
    'data',
    'models',
    'embeddings',
    'results/baseline_template',
    'results/baseline_zeroshot',
    'results/fewshot',
    'results/cot',
    'results/rag',
    'results/finetuned',
    'plots'
]

for folder in subfolders:
    os.makedirs(os.path.join(BASE_PATH, folder), exist_ok=True)

print(f"Project folder structure created at: {BASE_PATH}")
print()
print("Folder structure:")
for folder in subfolders:
    print(f"   {folder}/")

Mounted at /content/drive
Project folder structure created at: /content/drive/MyDrive/NLQuery

Folder structure:
   data/
   models/
   embeddings/
   results/baseline_template/
   results/baseline_zeroshot/
   results/fewshot/
   results/cot/
   results/rag/
   results/finetuned/
   plots/


---
## 1.4 Helper: Cache Manager
Saves and loads expensive outputs to/from Drive so you don't re-run them.

In [4]:
class CacheManager:
    def __init__(self, base_path):
        self.base_path = base_path

    def cache_path(self, subfolder, filename):
        return os.path.join(self.base_path, subfolder, filename)

    def exists(self, subfolder, filename):
        return os.path.exists(self.cache_path(subfolder, filename))

    def save_pickle(self, data, subfolder, filename):
        path = self.cache_path(subfolder, filename)
        with open(path, 'wb') as f:
            pickle.dump(data, f)
        print(f"Saved to Drive: {subfolder}/{filename}")

    def load_pickle(self, subfolder, filename):
        path = self.cache_path(subfolder, filename)
        with open(path, 'rb') as f:
            data = pickle.load(f)
        print(f"Loaded from Drive: {subfolder}/{filename}")
        return data

    def save_json(self, data, subfolder, filename):
        path = self.cache_path(subfolder, filename)
        with open(path, 'w') as f:
            json.dump(data, f, indent=2)
        print(f"Saved to Drive: {subfolder}/{filename}")

    def load_json(self, subfolder, filename):
        path = self.cache_path(subfolder, filename)
        with open(path, 'r') as f:
            data = json.load(f)
        print(f"Loaded from Drive: {subfolder}/{filename}")
        return data

    def save_csv(self, df, subfolder, filename):
        path = self.cache_path(subfolder, filename)
        df.to_csv(path, index=False)
        print(f"Saved to Drive: {subfolder}/{filename}")

    def load_csv(self, subfolder, filename):
        path = self.cache_path(subfolder, filename)
        df = pd.read_csv(path)
        print(f"Loaded from Drive: {subfolder}/{filename}")
        return df

cache = CacheManager(BASE_PATH)
print("Cache manager ready!")

Cache manager ready!


---
## 1.5 Load WikiTableQuestions Dataset

In [5]:
print("Loading WikiTableQuestions from HuggingFace...")
raw_dataset = load_dataset("wikitablequestions")

print()
print("Dataset loaded!")
print(f"   Train: {len(raw_dataset['train'])} examples")
print(f"   Test:  {len(raw_dataset['test'])} examples")
print(f"   Total: {len(raw_dataset['train']) + len(raw_dataset['test'])} examples")

Loading WikiTableQuestions from HuggingFace...


README.md: 0.00B [00:00, ?B/s]

wikitablequestions.py: 0.00B [00:00, ?B/s]

RuntimeError: Dataset scripts are no longer supported, but found wikitablequestions.py

In [ ]:
print("Dataset columns:", raw_dataset['train'].column_names)
print()
print("Sample entry:")
sample = raw_dataset['train'][0]
print(f"   Question: {sample['question']}")
print(f"   Answers:  {sample['answers']}")
print(f"   Table columns: {sample['table']['header']}")
print(f"   Table rows: {len(sample['table']['rows'])} rows")
print()
print("   First 3 rows:")
for i, row in enumerate(sample['table']['rows'][:3]):
    print(f"   Row {i}: {row}")

---
## 1.6 Process Tables to Pandas DataFrames

Key steps:
1. Convert each table to a Pandas DataFrame
2. Clean column names for SQL compatibility
3. Infer data types (numeric vs string)
4. Create SQLite databases for SQL execution

In [ ]:
def clean_column_name(col_name):
    cleaned = re.sub(r'[^a-zA-Z0-9_]', '_', col_name.strip())
    cleaned = cleaned.strip('_')
    cleaned = re.sub(r'_+', '_', cleaned)
    if cleaned and cleaned[0].isdigit():
        cleaned = 'col_' + cleaned
    if not cleaned:
        cleaned = 'unnamed_col'
    return cleaned.lower()


def try_convert_numeric(value):
    if pd.isna(value) or value == '' or value is None:
        return None
    cleaned = str(value).replace(',', '').replace('$', '').replace('%', '').strip()
    try:
        if '.' not in cleaned:
            return int(cleaned)
        return float(cleaned)
    except (ValueError, TypeError):
        return value


def table_to_dataframe(table_data):
    header = table_data['header']
    rows = table_data['rows']

    clean_cols = [clean_column_name(col) for col in header]

    seen = {}
    unique_cols = []
    for col in clean_cols:
        if col in seen:
            seen[col] += 1
            unique_cols.append(f"{col}_{seen[col]}")
        else:
            seen[col] = 0
            unique_cols.append(col)

    df = pd.DataFrame(rows, columns=unique_cols)

    for col in df.columns:
        converted = df[col].apply(try_convert_numeric)
        numeric_count = sum(1 for v in converted if isinstance(v, (int, float)) and v is not None)
        non_null_count = sum(1 for v in converted if v is not None)
        if non_null_count > 0 and numeric_count / non_null_count > 0.5:
            df[col] = pd.to_numeric(converted, errors='coerce')

    return df, header, unique_cols


print("Table processing functions defined!")

In [ ]:
sample = raw_dataset['train'][0]
df_test, orig_cols, clean_cols = table_to_dataframe(sample['table'])

print("Original columns:", orig_cols)
print("Cleaned columns: ", clean_cols)
print(f"DataFrame shape: {df_test.shape}")
print()
print("Data types:")
print(df_test.dtypes)
print()
print("Preview:")
display(df_test.head())

---
## 1.7 Build Processed Dataset with Table Metadata

For each question-answer pair, we store:
- The question and gold answer(s)
- The table as a DataFrame
- Original and cleaned column names
- Table schema info (column types, sample values)
- A unique table ID for SQLite lookup

In [ ]:
def process_split(dataset_split, split_name):
    processed_examples = []
    tables = {}

    for idx in tqdm(range(len(dataset_split)), desc=f"Processing {split_name}"):
        item = dataset_split[idx]
        question = item['question']
        answers = item['answers']
        table_data = item['table']

        table_id = f"{split_name}_{idx}"
        table_id = re.sub(r'[^a-zA-Z0-9_]', '_', table_id)

        if table_id not in tables:
            df, orig_cols, clean_cols = table_to_dataframe(table_data)

            schema = []
            for orig, clean in zip(orig_cols, clean_cols):
                col_type = str(df[clean].dtype) if clean in df.columns else 'unknown'
                sample_vals = df[clean].dropna().head(3).tolist() if clean in df.columns else []
                schema.append({
                    'original_name': orig,
                    'clean_name': clean,
                    'dtype': col_type,
                    'sample_values': [str(v) for v in sample_vals]
                })

            tables[table_id] = {
                'df': df,
                'orig_cols': orig_cols,
                'clean_cols': clean_cols,
                'schema': schema,
                'num_rows': len(df),
                'num_cols': len(clean_cols)
            }

        processed_examples.append({
            'id': f"{split_name}_{idx}",
            'question': question,
            'answers': answers,
            'table_id': table_id,
            'split': split_name
        })

    return processed_examples, tables


print("Processing function defined!")

In [ ]:
if cache.exists('data', 'processed_train.pkl') and cache.exists('data', 'processed_test.pkl'):
    print("Loading processed data from Drive cache...")
    train_data = cache.load_pickle('data', 'processed_train.pkl')
    test_data = cache.load_pickle('data', 'processed_test.pkl')
    train_examples = train_data['examples']
    train_tables = train_data['tables']
    test_examples = test_data['examples']
    test_tables = test_data['tables']
else:
    print("Processing dataset (first time only)...")
    train_examples, train_tables = process_split(raw_dataset['train'], 'train')
    test_examples, test_tables = process_split(raw_dataset['test'], 'test')

    cache.save_pickle({'examples': train_examples, 'tables': train_tables}, 'data', 'processed_train.pkl')
    cache.save_pickle({'examples': test_examples, 'tables': test_tables}, 'data', 'processed_test.pkl')

print()
print("Processing complete!")
print(f"   Train: {len(train_examples)} examples, {len(train_tables)} unique tables")
print(f"   Test:  {len(test_examples)} examples, {len(test_tables)} unique tables")

---
## 1.8 Create SQLite Databases

Each table gets stored in a SQLite database so we can execute generated SQL queries against it.

In [ ]:
def create_sqlite_db(tables_dict, db_path):
    conn = sqlite3.connect(db_path)
    success_count = 0
    error_count = 0

    for table_id, table_info in tqdm(tables_dict.items(), desc="Creating SQLite tables"):
        try:
            df = table_info['df']
            df.to_sql(table_id, conn, if_exists='replace', index=False)
            success_count += 1
        except Exception as e:
            error_count += 1
            if error_count <= 5:
                print(f"   Warning: {table_id}: {str(e)[:100]}")

    conn.close()
    print(f"SQLite DB created: {success_count} tables loaded, {error_count} errors")
    return success_count, error_count


print("SQLite creation function defined!")

In [ ]:
TRAIN_DB_PATH = cache.cache_path('data', 'train_tables.db')
TEST_DB_PATH = cache.cache_path('data', 'test_tables.db')

if os.path.exists(TRAIN_DB_PATH) and os.path.exists(TEST_DB_PATH):
    print("SQLite databases already exist on Drive. Skipping creation.")
else:
    print("Creating SQLite databases (first time only)...")
    print("--- Train DB ---")
    create_sqlite_db(train_tables, TRAIN_DB_PATH)
    print("--- Test DB ---")
    create_sqlite_db(test_tables, TEST_DB_PATH)

print()
print(f"Train DB: {TRAIN_DB_PATH}")
print(f"Test DB:  {TEST_DB_PATH}")

---
## 1.9 SQL Execution Engine

Core function used across ALL approaches - takes generated SQL, runs it, returns result.

In [ ]:
def execute_sql(sql_query, table_id, db_path, timeout=10):
    try:
        conn = sqlite3.connect(db_path)
        conn.execute("PRAGMA busy_timeout = " + str(timeout * 1000))

        modified_sql = sql_query
        for placeholder in ['FROM table', 'FROM w', 'FROM t ', 'JOIN table', 'JOIN w', 'JOIN t ']:
            if placeholder.lower() in modified_sql.lower():
                replacement = placeholder.split()[0] + ' "' + table_id + '"'
                modified_sql = re.sub(
                    re.escape(placeholder),
                    replacement,
                    modified_sql,
                    flags=re.IGNORECASE
                )

        cursor = conn.execute(modified_sql)
        rows = cursor.fetchall()
        conn.close()

        if not rows:
            return [], True, None

        if len(rows[0]) == 1:
            result = [str(row[0]) for row in rows]
        else:
            result = [str(val) for row in rows for val in row]

        return result, True, None

    except Exception as e:
        return [], False, str(e)


print("SQL execution engine ready!")

In [ ]:
test_example = test_examples[0]
test_table_id = test_example['table_id']
test_table_info = test_tables[test_table_id]

print(f"Question: {test_example['question']}")
print(f"Gold Answer: {test_example['answers']}")
print(f"Table ID: {test_table_id}")
print(f"Columns: {test_table_info['clean_cols']}")
print(f"Rows: {test_table_info['num_rows']}")
print()
print("Table preview:")
display(test_table_info['df'].head())

simple_sql = 'SELECT * FROM "' + test_table_id + '" LIMIT 3'
result, success, error = execute_sql(simple_sql, test_table_id, TEST_DB_PATH)
print()
print(f"Test query: {simple_sql}")
print(f"   Success: {success}")
print(f"   Result: {result[:10]}")

---
## 1.10 Evaluation Framework

Core metric: **Execution Accuracy** - does the SQL result match the gold answer?

In [ ]:
def normalize_answer(answer):
    if answer is None:
        return ''
    answer = str(answer).strip().lower()
    if answer.endswith('.0'):
        answer = answer[:-2]
    answer = answer.replace(',', '')
    return answer


def check_answer(predicted_results, gold_answers):
    if not predicted_results:
        return False, 'empty'

    pred_normalized = [normalize_answer(p) for p in predicted_results]
    gold_normalized = [normalize_answer(g) for g in gold_answers]

    if sorted(pred_normalized) == sorted(gold_normalized):
        return True, 'exact'

    for pred in pred_normalized:
        if pred in gold_normalized:
            return True, 'subset'

    if len(pred_normalized) >= 1 and len(gold_normalized) == 1:
        if pred_normalized[0] == gold_normalized[0]:
            return True, 'exact'

    return False, 'none'


def evaluate_predictions(predictions, examples, db_path, tables_dict):
    example_lookup = {ex['id']: ex for ex in examples}

    total = len(predictions)
    correct = 0
    sql_valid = 0
    errors = []
    details = []

    for pred in tqdm(predictions, desc="Evaluating"):
        example = example_lookup[pred['id']]
        table_id = example['table_id']
        gold_answers = example['answers']

        result, success, error = execute_sql(pred['sql'], table_id, db_path)

        if success:
            sql_valid += 1
            match, match_type = check_answer(result, gold_answers)
            if match:
                correct += 1
        else:
            match = False
            match_type = 'sql_error'
            errors.append({'id': pred['id'], 'error': error, 'sql': pred['sql']})

        details.append({
            'id': pred['id'],
            'question': example['question'],
            'gold_answers': gold_answers,
            'predicted_sql': pred['sql'],
            'predicted_result': result if success else None,
            'sql_valid': success,
            'correct': match,
            'match_type': match_type,
            'error': error,
            'latency': pred.get('latency', None)
        })

    has_latency = any(d['latency'] is not None for d in details)
    avg_lat = float(np.mean([d['latency'] for d in details if d['latency'] is not None])) if has_latency else 0.0

    metrics = {
        'total': total,
        'execution_accuracy': correct / total if total > 0 else 0,
        'sql_validity_rate': sql_valid / total if total > 0 else 0,
        'correct_count': correct,
        'sql_valid_count': sql_valid,
        'avg_latency': avg_lat,
        'error_count': len(errors)
    }

    return metrics, details


print("Evaluation framework ready!")

---
## 1.11 Query Complexity Classification

Classify each question into Simple / Medium / Complex based on linguistic patterns.

In [ ]:
def classify_complexity(question, answers, table_info):
    q = question.lower()
    score = 0

    medium_keywords = ['how many', 'total', 'count', 'average', 'sum',
                       'more than', 'less than', 'between', 'before', 'after',
                       'greater', 'at least', 'maximum', 'minimum']

    complex_keywords = ['difference', 'compared to', 'both', 'and also',
                        'highest', 'lowest', 'most', 'least', 'rank',
                        'same as', 'different from', 'top', 'bottom',
                        'first and last', 'how many more', 'how many fewer',
                        'percentage', 'ratio', 'consecutive']

    for kw in medium_keywords:
        if kw in q:
            score += 1

    for kw in complex_keywords:
        if kw in q:
            score += 2

    if len(answers) > 1:
        score += 1
    if len(question.split()) > 15:
        score += 1
    if table_info['num_rows'] > 20:
        score += 1

    if score <= 1:
        return 'simple'
    elif score <= 3:
        return 'medium'
    else:
        return 'complex'


print("Complexity classifier defined!")

In [ ]:
if cache.exists('data', 'complexity_labels.json'):
    complexity_labels = cache.load_json('data', 'complexity_labels.json')
else:
    print("Classifying question complexity...")
    complexity_labels = {}

    for ex in tqdm(train_examples + test_examples, desc="Classifying"):
        split_name = ex['split']
        tables_ref = train_tables if split_name == 'train' else test_tables
        table_info = tables_ref[ex['table_id']]
        complexity = classify_complexity(ex['question'], ex['answers'], table_info)
        complexity_labels[ex['id']] = complexity

    cache.save_json(complexity_labels, 'data', 'complexity_labels.json')

complexity_counts = Counter(complexity_labels.values())
print()
print("Complexity Distribution:")
for level in ['simple', 'medium', 'complex']:
    count = complexity_counts.get(level, 0)
    pct = count / len(complexity_labels) * 100
    print(f"   {level.capitalize():10s}: {count:5d} ({pct:.1f}%)")

---
## 1.12 Create Evaluation Subset

Stratified subset of 500 test examples for quick iteration.

In [ ]:
def create_stratified_subset(examples, complexity_labels, n=500, seed=42):
    np.random.seed(seed)

    groups = defaultdict(list)
    for ex in examples:
        complexity = complexity_labels.get(ex['id'], 'simple')
        groups[complexity].append(ex)

    total = len(examples)
    subset = []
    for level, group_examples in groups.items():
        k = max(1, int(n * len(group_examples) / total))
        k = min(k, len(group_examples))
        indices = np.random.choice(len(group_examples), k, replace=False)
        subset.extend([group_examples[i] for i in indices])

    np.random.shuffle(subset)
    return subset[:n]


if cache.exists('data', 'eval_subset_500.json'):
    eval_subset = cache.load_json('data', 'eval_subset_500.json')
else:
    eval_subset = create_stratified_subset(test_examples, complexity_labels, n=500)
    cache.save_json(eval_subset, 'data', 'eval_subset_500.json')

subset_complexity = Counter(complexity_labels.get(ex['id'], 'simple') for ex in eval_subset)
print(f"Quick Eval Subset ({len(eval_subset)} examples):")
for level in ['simple', 'medium', 'complex']:
    count = subset_complexity.get(level, 0)
    print(f"   {level.capitalize():10s}: {count}")

---
## 1.13 Schema Formatter (Used by All Approaches)

Creates a text representation of a table schema for LLM prompts.

In [ ]:
def format_schema(table_info, include_samples=True, max_sample_rows=3):
    lines = []
    lines.append("Table has " + str(table_info['num_rows']) + " rows and " + str(table_info['num_cols']) + " columns.")
    lines.append("Columns:")

    for col_info in table_info['schema']:
        dtype = 'text' if 'object' in col_info['dtype'] else 'numeric'
        samples = ', '.join(col_info['sample_values'][:3])
        lines.append("  - " + col_info['clean_name'] + " (" + dtype + ") | Examples: " + samples)

    if include_samples:
        lines.append("")
        lines.append("Sample rows (first " + str(max_sample_rows) + "):")
        df = table_info['df']
        sample_df = df.head(max_sample_rows)
        lines.append(sample_df.to_string(index=False))

    return '\n'.join(lines)


sample_table_id = test_examples[0]['table_id']
sample_table_info = test_tables[sample_table_id]
print("Schema format example:")
print("=" * 60)
print(format_schema(sample_table_info))
print("=" * 60)

---
## 1.14 GPU Check and Environment Info

In [ ]:
import torch

print("Environment Info:")
print("   Python:", sys.version.split()[0])
print("   PyTorch:", torch.__version__)
print("   CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("   GPU:", torch.cuda.get_device_name(0))
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print("   GPU Memory: {:.1f} GB".format(gpu_mem))
else:
    print("   No GPU detected.")

---
## Section 1 Complete!

### What is now ready:
1. All dependencies installed
2. Google Drive mounted with folder structure
3. Cache manager for saving/loading expensive outputs
4. WikiTableQuestions dataset loaded and processed
5. Tables converted to DataFrames with clean column names
6. SQLite databases created for SQL execution
7. SQL execution engine ready
8. Evaluation framework (execution accuracy, SQL validity)
9. Query complexity classifier (Simple/Medium/Complex)
10. Stratified eval subset (500 examples for quick testing)
11. Schema formatter for LLM prompts
12. GPU environment verified

### Cached on Drive (will not re-run next session):
- data/processed_train.pkl
- data/processed_test.pkl
- data/train_tables.db
- data/test_tables.db
- data/complexity_labels.json
- data/eval_subset_500.json

### Next: Section 2 - Template-Based SQL Baseline

## Section 2: Template-Based SQL Baseline (Non-LLM)

This is our first baseline — a rule-based system that converts natural language questions to SQL using keyword matching and patterns. No LLM involved. This sets the floor for comparison.

In [ ]:
def template_based_sql(question, table_info):
    """
    Generate SQL from natural language using keyword matching rules.
    No LLM — purely rule-based.
    """
    q = question.lower().strip()
    table_name = "t"  # placeholder, will be replaced by execute_sql
    columns = table_info['clean_cols']
    schema = table_info['schema']

    # Try to find which column the question is about
    def find_relevant_column(q, columns, schema):
        """Match question words to column names."""
        best_col = None
        best_score = 0
        for i, col_info in enumerate(schema):
            score = 0
            orig_name = col_info['original_name'].lower()
            clean_name = col_info['clean_name'].lower()
            # Check if original column name appears in question
            for word in orig_name.split():
                if len(word) > 2 and word in q:
                    score += 2
            for word in clean_name.split('_'):
                if len(word) > 2 and word in q:
                    score += 1
            if score > best_score:
                best_score = score
                best_col = columns[i]
        return best_col, best_score

    # Find the column most relevant to the question
    target_col, col_score = find_relevant_column(q, columns, schema)

    # Find a filter column and value (look for quoted values or proper nouns)
    def find_filter(q, columns, schema, table_info):
        """Try to find a WHERE clause from the question."""
        df = table_info['df']
        best_filter_col = None
        best_filter_val = None
        best_match_len = 0

        for col in columns:
            try:
                unique_vals = df[col].dropna().astype(str).unique()
                for val in unique_vals:
                    val_lower = val.lower()
                    if len(val_lower) > 2 and val_lower in q:
                        if len(val_lower) > best_match_len:
                            best_match_len = len(val_lower)
                            best_filter_col = col
                            best_filter_val = val
            except:
                continue

        return best_filter_col, best_filter_val

    filter_col, filter_val = find_filter(q, columns, schema, table_info)

    # Determine query type based on keywords
    sql = None

    # COUNT queries
    if any(kw in q for kw in ['how many', 'count', 'number of', 'total number']):
        if filter_col and filter_val:
            sql = f'SELECT COUNT(*) FROM t WHERE "{filter_col}" = \'{filter_val}\''
        elif target_col:
            sql = f'SELECT COUNT(DISTINCT "{target_col}") FROM t'
        else:
            sql = 'SELECT COUNT(*) FROM t'

    # SUM queries
    elif any(kw in q for kw in ['total', 'sum', 'combined', 'altogether']):
        if target_col:
            sql = f'SELECT SUM("{target_col}") FROM t'
        else:
            # Try to find a numeric column
            for col_info in schema:
                if 'int' in col_info['dtype'] or 'float' in col_info['dtype']:
                    sql = f'SELECT SUM("{col_info["clean_name"]}") FROM t'
                    break

    # AVERAGE queries
    elif any(kw in q for kw in ['average', 'avg', 'mean']):
        if target_col:
            sql = f'SELECT AVG("{target_col}") FROM t'
        else:
            for col_info in schema:
                if 'int' in col_info['dtype'] or 'float' in col_info['dtype']:
                    sql = f'SELECT AVG("{col_info["clean_name"]}") FROM t'
                    break

    # MAX/HIGHEST queries
    elif any(kw in q for kw in ['most', 'highest', 'largest', 'greatest', 'maximum', 'max', 'best', 'top']):
        if target_col:
            if filter_col and filter_col != target_col:
                sql = f'SELECT "{filter_col}" FROM t ORDER BY "{target_col}" DESC LIMIT 1'
            else:
                sql = f'SELECT MAX("{target_col}") FROM t'
        else:
            sql = f'SELECT * FROM t ORDER BY "{columns[-1]}" DESC LIMIT 1'

    # MIN/LOWEST queries
    elif any(kw in q for kw in ['least', 'lowest', 'smallest', 'minimum', 'min', 'worst', 'fewest']):
        if target_col:
            if filter_col and filter_col != target_col:
                sql = f'SELECT "{filter_col}" FROM t ORDER BY "{target_col}" ASC LIMIT 1'
            else:
                sql = f'SELECT MIN("{target_col}") FROM t'
        else:
            sql = f'SELECT * FROM t ORDER BY "{columns[-1]}" ASC LIMIT 1'

    # LAST/LATEST queries
    elif any(kw in q for kw in ['last', 'latest', 'most recent', 'final']):
        if target_col:
            sql = f'SELECT "{target_col}" FROM t ORDER BY ROWID DESC LIMIT 1'
        elif filter_col:
            sql = f'SELECT "{filter_col}" FROM t ORDER BY ROWID DESC LIMIT 1'
        else:
            sql = f'SELECT * FROM t ORDER BY ROWID DESC LIMIT 1'

    # FIRST/EARLIEST queries
    elif any(kw in q for kw in ['first', 'earliest', 'initial', 'beginning']):
        if target_col:
            sql = f'SELECT "{target_col}" FROM t ORDER BY ROWID ASC LIMIT 1'
        elif filter_col:
            sql = f'SELECT "{filter_col}" FROM t ORDER BY ROWID ASC LIMIT 1'
        else:
            sql = f'SELECT * FROM t ORDER BY ROWID ASC LIMIT 1'

    # Simple lookup with filter
    elif filter_col and filter_val and target_col:
        sql = f'SELECT "{target_col}" FROM t WHERE "{filter_col}" = \'{filter_val}\''

    # Fallback: just select the most relevant column
    elif target_col:
        sql = f'SELECT "{target_col}" FROM t LIMIT 1'

    # Ultimate fallback
    if sql is None:
        sql = f'SELECT "{columns[0]}" FROM t LIMIT 1'

    return sql


print("Template SQL generator defined!")

##  Test Template on Examples:

In [ ]:
# Test on a few examples to see what SQL it generates
print("Template SQL Examples:")
print("=" * 70)

for i in range(5):
    ex = test_examples[i]
    table_info = test_tables[ex['table_id']]
    sql = template_based_sql(ex['question'], table_info)
    result, success, error = execute_sql(sql, ex['table_id'], TEST_DB_PATH)

    print(f"\nQ: {ex['question']}")
    print(f"   Generated SQL: {sql}")
    print(f"   Result: {result[:3] if result else 'empty'}")
    print(f"   Gold Answer: {ex['answers']}")
    match, _ = check_answer(result, ex['answers'])
    print(f"   Correct: {match}")
    print("-" * 70)

### Run Template Baseline on Eval Subset:

In [ ]:
# Run template baseline on the 500-example eval subset
print("Running Template SQL baseline on eval subset...")

template_predictions = []
for ex in tqdm(eval_subset, desc="Template SQL"):
    table_info = test_tables[ex['table_id']]

    start_time = time.time()
    sql = template_based_sql(ex['question'], table_info)
    latency = time.time() - start_time

    template_predictions.append({
        'id': ex['id'],
        'sql': sql,
        'latency': latency
    })

print(f"Generated {len(template_predictions)} SQL queries")

### Evaluate Template Baseline

In [ ]:
# Evaluate
template_metrics, template_details = evaluate_predictions(
    template_predictions, eval_subset, TEST_DB_PATH, test_tables
)

print()
print("=" * 50)
print("TEMPLATE SQL BASELINE RESULTS")
print("=" * 50)
print(f"   Execution Accuracy: {template_metrics['execution_accuracy']:.1%}")
print(f"   SQL Validity Rate:  {template_metrics['sql_validity_rate']:.1%}")
print(f"   Correct: {template_metrics['correct_count']} / {template_metrics['total']}")
print(f"   Avg Latency: {template_metrics['avg_latency']*1000:.1f} ms")
print(f"   Errors: {template_metrics['error_count']}")
print("=" * 50)

### Breakdown by Complexity

In [ ]:
# Breakdown by complexity
print("Results by Complexity:")
print("-" * 50)

for level in ['simple', 'medium', 'complex']:
    level_details = [d for d in template_details if complexity_labels.get(d['id']) == level]
    if level_details:
        correct = sum(1 for d in level_details if d['correct'])
        total = len(level_details)
        print(f"   {level.capitalize():10s}: {correct}/{total} = {correct/total:.1%}")

### Error Analysis

In [ ]:
# Error analysis - what types of errors does template SQL make?
from collections import Counter

error_types = []
for d in template_details:
    if not d['correct']:
        if d['match_type'] == 'sql_error':
            error_types.append('SQL Error')
        elif d['match_type'] == 'empty':
            error_types.append('Empty Result')
        else:
            error_types.append('Wrong Answer')

error_counts = Counter(error_types)
print("Error Breakdown:")
print("-" * 40)
for error_type, count in error_counts.most_common():
    print(f"   {error_type:15s}: {count} ({count/len(template_details):.1%})")

### Save Results to Drive

In [ ]:
# Save predictions and metrics to Drive
cache.save_json(template_predictions, 'results/baseline_template', 'predictions.json')
cache.save_json(template_metrics, 'results/baseline_template', 'metrics.json')
cache.save_json(template_details, 'results/baseline_template', 'details.json')

print()
print("Section 2 complete! Template baseline saved to Drive.")
print("This is our floor - every LLM approach should beat this.")

## Section 3: Zero-Shot LLM Baseline (Qwen2.5-7B-coder)

First LLM approach — give the model the table schema and question, ask it to generate SQL with no examples. This tests the model's raw ability to understand natural language and produce valid SQL.

### Load Qwen2.5-coder-7B_Instruct

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_NAME} in 4-bit quantization...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

print()
print("Model loaded!")
print(f"   Model: {MODEL_NAME}")
print(f"   GPU Memory Used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

### Define Generation Function

In [ ]:
def generate_sql(prompt, model, tokenizer, max_new_tokens=256):
    """
    Generate SQL from a prompt using the loaded model.
    Returns the generated text and latency.
    """
    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False  # Disable thinking mode for clean SQL output
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    start_time = time.time()
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            top_p=0.95,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    latency = time.time() - start_time

    # Extract only the new tokens (not the prompt)
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
    response = tokenizer.decode(output_ids, skip_special_tokens=True)

    return response, latency


print("Generation function defined!")

### Define Zero-Shot Prompt Template

In [ ]:
def build_zeroshot_prompt(question, table_info):
    """
    Build a zero-shot prompt for SQL generation.
    Gives the model the table schema and asks for SQL.
    """
    schema_text = format_schema(table_info, include_samples=True, max_sample_rows=3)

    prompt = f"""You are a SQL expert. Given a table and a question, generate a SQLite query to answer the question.

{schema_text}

Question: {question}

Rules:
- Return ONLY the SQL query, nothing else
- Use the exact column names provided above
- The table name is "t"
- Use SQLite syntax

SQL:"""

    return prompt


# Test the prompt
sample_ex = test_examples[0]
sample_table = test_tables[sample_ex['table_id']]
sample_prompt = build_zeroshot_prompt(sample_ex['question'], sample_table)
print("Sample Zero-Shot Prompt:")
print("=" * 60)
print(sample_prompt)
print("=" * 60)

### Test on a Few Examples

In [ ]:
# Test on 5 examples to make sure everything works
print("Testing zero-shot generation on 5 examples...")
print("=" * 70)

for i in range(5):
    ex = test_examples[i]
    table_info = test_tables[ex['table_id']]
    prompt = build_zeroshot_prompt(ex['question'], table_info)

    response, latency = generate_sql(prompt, model, tokenizer)

    # Extract SQL from response (take first line that looks like SQL)
    sql = response.strip().split('\n')[0].strip()
    # Remove markdown code blocks if present
    sql = sql.replace('```sql', '').replace('```', '').strip()

    result, success, error = execute_sql(sql, ex['table_id'], TEST_DB_PATH)
    match, _ = check_answer(result, ex['answers'])

    print(f"\nQ: {ex['question']}")
    print(f"   Generated SQL: {sql}")
    print(f"   Result: {result[:3] if result else 'empty/error'}")
    print(f"   Gold Answer: {ex['answers']}")
    print(f"   Correct: {match} | Latency: {latency:.2f}s")
    print("-" * 70)

### SQL Extraction Helper

In [ ]:
def extract_sql_from_response(response):
    """
    Extract clean SQL from model response.
    Handles various output formats (markdown blocks, explanations, etc.)
    """
    text = response.strip()

    # Remove markdown code blocks
    if '```sql' in text:
        match = re.search(r'```sql\s*(.*?)\s*```', text, re.DOTALL)
        if match:
            return match.group(1).strip()

    if '```' in text:
        match = re.search(r'```\s*(.*?)\s*```', text, re.DOTALL)
        if match:
            return match.group(1).strip()

    # Take first line that starts with SELECT, WITH, etc.
    for line in text.split('\n'):
        line = line.strip()
        if line.upper().startswith(('SELECT', 'WITH', 'INSERT', 'UPDATE', 'DELETE')):
            # Collect multi-line SQL
            sql_lines = [line]
            remaining = text.split('\n')
            start_idx = remaining.index(line) if line in remaining else -1
            if start_idx >= 0:
                for next_line in remaining[start_idx + 1:]:
                    next_line = next_line.strip()
                    if next_line and not next_line.startswith(('--', '#', 'Note', 'This', 'The', 'Here')):
                        sql_lines.append(next_line)
                        if next_line.endswith(';'):
                            break
                    else:
                        break
            return ' '.join(sql_lines).rstrip(';').strip()

    # Fallback: return first non-empty line
    for line in text.split('\n'):
        if line.strip():
            return line.strip()

    return text


print("SQL extraction helper defined!")

Run Zero-Shot on Eval Subset

In [ ]:
# Run zero-shot on the 500-example eval subset
print("Running zero-shot Qwen3.5-9B on eval subset...")
print("This will take approximately 30-60 minutes...")

zeroshot_predictions = []
errors_during_generation = 0

for i, ex in enumerate(tqdm(eval_subset, desc="Zero-shot")):
    table_info = test_tables[ex['table_id']]
    prompt = build_zeroshot_prompt(ex['question'], table_info)

    try:
        response, latency = generate_sql(prompt, model, tokenizer)
        sql = extract_sql_from_response(response)
    except Exception as e:
        sql = "SELECT 1"
        latency = 0
        errors_during_generation += 1

    zeroshot_predictions.append({
        'id': ex['id'],
        'sql': sql,
        'latency': latency
    })

    # Print progress every 100 examples
    if (i + 1) % 100 == 0:
        print(f"   Completed {i+1}/{len(eval_subset)} examples")

print(f"\nGeneration complete! Errors during generation: {errors_during_generation}")

### Evaluate Zero-Shot

In [ ]:
# Evaluate zero-shot results
zeroshot_metrics, zeroshot_details = evaluate_predictions(
    zeroshot_predictions, eval_subset, TEST_DB_PATH, test_tables
)

print()
print("=" * 50)
print("ZERO-SHOT QWEN3.5-9B RESULTS")
print("=" * 50)
print(f"   Execution Accuracy: {zeroshot_metrics['execution_accuracy']:.1%}")
print(f"   SQL Validity Rate:  {zeroshot_metrics['sql_validity_rate']:.1%}")
print(f"   Correct: {zeroshot_metrics['correct_count']} / {zeroshot_metrics['total']}")
print(f"   Avg Latency: {zeroshot_metrics['avg_latency']:.2f}s")
print(f"   Errors: {zeroshot_metrics['error_count']}")
print("=" * 50)

print()
print("Comparison with Template Baseline:")
print(f"   Template:  {template_metrics['execution_accuracy']:.1%}")
print(f"   Zero-shot: {zeroshot_metrics['execution_accuracy']:.1%}")
improvement = zeroshot_metrics['execution_accuracy'] - template_metrics['execution_accuracy']
print(f"   Improvement: +{improvement:.1%}")

### Complexity Breakdown

In [ ]:
# Breakdown by complexity
print("Zero-Shot Results by Complexity:")
print("-" * 50)

for level in ['simple', 'medium', 'complex']:
    level_details = [d for d in zeroshot_details if complexity_labels.get(d['id']) == level]
    if level_details:
        correct = sum(1 for d in level_details if d['correct'])
        total = len(level_details)
        print(f"   {level.capitalize():10s}: {correct}/{total} = {correct/total:.1%}")

### Error Analysis

In [ ]:
# Error analysis
error_types = []
for d in zeroshot_details:
    if not d['correct']:
        if d['match_type'] == 'sql_error':
            error_types.append('SQL Error')
        elif d['match_type'] == 'empty':
            error_types.append('Empty Result')
        else:
            error_types.append('Wrong Answer')

error_counts = Counter(error_types)
print("Zero-Shot Error Breakdown:")
print("-" * 40)
for error_type, count in error_counts.most_common():
    print(f"   {error_type:15s}: {count} ({count/len(zeroshot_details):.1%})")

### Save Results

In [ ]:
# Save to Drive
cache.save_json(zeroshot_predictions, 'results/baseline_zeroshot', 'predictions.json')
cache.save_json(zeroshot_metrics, 'results/baseline_zeroshot', 'metrics.json')
cache.save_json(zeroshot_details, 'results/baseline_zeroshot', 'details.json')

print()
print("Section 3 complete! Zero-shot results saved to Drive.")
print("Next: Section 4 - Few-shot & Chain-of-Thought prompting")